# PALABRIA – Notebook de ejecución

Este notebook permite ejecutar la aplicación PALABRIA de forma sencilla en Google Colab, y exponerlo mediante una URL pública.


Privacidad y datos:
- Cada usuario crea su propia base de datos.
- La base de datos NO se comparte con los autores del proyecto.


In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Ejecuta esta celda SOLO si ya tienes la carpeta aplicacion_PALABRIA en tu Drive

import os, sys

PROJECT_PATH = "/content/drive/MyDrive/Master/SemiCuatri3/IAE/Asistente-Educacion"

if not os.path.exists(PROJECT_PATH):
    raise FileNotFoundError(
        "No se encontró la carpeta 'aplicacion_PALABRIA' en tu Drive.\n"
        "Copia ahí el proyecto o usa la celda de GitHub."
    )

os.chdir(PROJECT_PATH)
sys.path.append(PROJECT_PATH)

In [3]:
# Ejecuta esta celda SOLO si vas a descargar el proyecto desde GitHub en el sistema TEMPORAL de Colab

import os, sys

PROJECT_PATH = "/content/drive/MyDrive/Master/SemiCuatri3/IAE/Asistente-Educacion"
REPO_URL = "https://github.com/ncenteno-arch/APP-PALABRIA.git"


if not os.path.exists(PROJECT_PATH):
    !git clone $REPO_URL $PROJECT_PATH

os.chdir(PROJECT_PATH)
sys.path.append(PROJECT_PATH)

In [4]:
# Define la base de datos SQL
# La BD se guarda en tú Google Drive, independientemente de si el código viene de Drive o de GitHub

import pathlib

DB_PATH = "/content/drive/MyDrive/Master/SemiCuatri3/IAE/Asistente-Educacion/data/palabria.db"
pathlib.Path(os.path.dirname(DB_PATH)).mkdir(parents=True, exist_ok=True)

os.environ["DB_PATH"] = DB_PATH

print("Base de datos personal configurada en:", DB_PATH)

Base de datos personal configurada en: /content/drive/MyDrive/Master/SemiCuatri3/IAE/Asistente-Educacion/data/palabria.db


In [5]:
!pip install -r requirements.txt

In [6]:
# Se descarga el modelo lingüístico de spaCy para español (necesario para la app)

!python -m spacy download es_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.0/568.0 MB 2.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
# Permite que el backend pueda quedarse funcionando en segundo plano, de forma que después podamos arrancar la interfaz web (frontend) sin que el notebook se bloquee

import nest_asyncio, logging

nest_asyncio.apply()
logging.basicConfig(level=logging.INFO)

In [17]:
# Se lanza el backend en local

import subprocess, time, os
import socket # For port checking

os.chdir(PROJECT_PATH)

# Function to check if a port is open
def is_port_open(host, port, timeout=1):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(timeout)
    try:
        sock.connect((host, port))
        return True
    except (socket.error, socket.timeout):
        return False
    finally:
        sock.close()

backend_process = subprocess.Popen(
    [
        "uvicorn",
        "backend.main:app",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--log-level", "warning"
    ]
)

# Wait until the backend port is open
timeout_seconds = 60
start_time = time.time()
print("Esperando a que el backend se inicie...")
while not is_port_open("127.0.0.1", 8000):
    if time.time() - start_time > timeout_seconds:
        print("Error: El backend no se inició a tiempo.")
        backend_process.terminate()
        raise RuntimeError("Backend falló al iniciar.")
    time.sleep(1)

print("Backend activo")

Esperando a que el backend se inicie...
Backend activo


In [18]:
# El frontend se conecta al backend a través de un túnel ngrok

import os
from pyngrok import ngrok
import time

# Iniciar el túnel ngrok para el backend en el puerto 8000
backend_tunnel = ngrok.connect(8000, bind_tls=True)
BACKEND_URL = backend_tunnel.public_url

os.environ["BACKEND_URL"] = BACKEND_URL

print("Backend disponible públicamente en:", BACKEND_URL)
print("Frontend configurado para conectar al backend")

# Esperar un poco para que el túnel ngrok se estabilice
time.sleep(5)

Backend disponible públicamente en: https://lumberingly-subtriquetrous-iesha.ngrok-free.dev
Frontend configurado para conectar al backend


In [10]:
# Introduce TU token de ngrok (ngrok crea una dirección web pública temporal)

from pyngrok import ngrok

NGROK_TOKEN = input("Pega aquí tu NGROK AUTHTOKEN (no se guarda): ").strip()
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

Pega aquí tu NGROK AUTHTOKEN (no se guarda): 39G81fo3cFIz5wfI5EdevdgMNip_3Efmg5iNfMEcZuh3TvETL


In [19]:
import subprocess
import sys
from pyngrok import ngrok

PORT = 8501  # puedes usar 8000 si prefieres

# Opcional: cerrar túneles previos
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

frontend_tunnel = ngrok.connect(PORT, bind_tls=True)

# Lanza el frontend Streamlit
streamlit_process = subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", "frontend/app.py",
    "--server.port", str(PORT),
    "--server.headless", "true",
    "--browser.serverAddress", "0.0.0.0"
])

print("APLICACIÓN DISPONIBLE EN:")
print(frontend_tunnel.public_url)

APLICACIÓN DISPONIBLE EN:
https://lumberingly-subtriquetrous-iesha.ngrok-free.dev


**Recomendación:** lo más fiable es reiniciar el entorno de Colab para volver a ejecutar la aplicación.
Si no quieres reiniciar, prueba primero a limpiar los procesos en segundo plano.

In [32]:
# Limpia los procesos anteriores (backend, frontend y ngrok)

import subprocess

subprocess.run(["pkill", "-f", "uvicorn"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "streamlit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "ngrok"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

CompletedProcess(args=['pkill', '-f', 'ngrok'], returncode=0)

In [33]:
import os, sys, time, subprocess
import requests
from pyngrok import ngrok
import socket # For port checking

PROJECT = r"/content/drive/MyDrive/Master/SemiCuatri3/IAE/Asistente-Educacion/"
PORT = 8000  # usa este mismo en uvicorn y ngrok

# Function to check if a port is open
def is_port_open(host, port, timeout=1):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(timeout)
    try:
        sock.connect((host, port))
        return True
    except (socket.error, socket.timeout):
        return False
    finally:
        sock.close()

# Cerrar túneles previos
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

# (Opcional) parar uvicorn previo
# subprocess.run(["taskkill", "/F", "/IM", "python.exe"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) # Esto es para Windows

# Levantar FastAPI
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=PROJECT
)

# Wait until the backend port is open
timeout_seconds = 60
start_time = time.time()
print("Esperando a que el backend se inicie...")
while not is_port_open("127.0.0.1", PORT):
    if time.time() - start_time > timeout_seconds:
        print("Error: El backend no se inició a tiempo.")
        proc.terminate()
        raise RuntimeError("Backend falló al iniciar.")
    time.sleep(1)

print("Backend activo")
print("STATUS local:", requests.get(f"http://127.0.0.1:{PORT}/status/").status_code)  # debe dar 200

tunnel = ngrok.connect(PORT, bind_tls=True)
print("APP:", tunnel.public_url)
print("DOCS:", tunnel.public_url + "/docs")

Esperando a que el backend se inicie...
Backend activo
STATUS local: 200
APP: https://lumberingly-subtriquetrous-iesha.ngrok-free.dev
DOCS: https://lumberingly-subtriquetrous-iesha.ngrok-free.dev/docs
